In [1]:
import numpy as np
import open3d as o3d

def assemble_pieces(piece_point_clouds, Ti0_list):
    """
    拼接多个碎片的点云，基于相对0号碎片的变换矩阵Ti0
    Args:
        piece_point_clouds (list): 每个元素是单个碎片的局部点云，shape=[N_i, 3]，共n个碎片
        Ti0_list (list): 每个元素是碎片i相对0号的4×4变换矩阵，shape=[4,4]，共n个碎片
    Returns:
        full_pcd (open3d.geometry.PointCloud): 拼接后的完整点云
        transformed_pieces (list): 每个碎片变换后的点云（全局坐标系），用于可视化对比
    """
    # 输入校验：碎片数量和变换矩阵数量一致
    n = len(piece_point_clouds)
    assert len(Ti0_list) == n, "碎片数量和变换矩阵数量不匹配"
    
    transformed_pieces = []
    for i in range(n):
        # 1. 提取当前碎片的局部点云（N_i, 3）
        local_pc = piece_point_clouds[i].copy()
        # 2. 提取变换矩阵的旋转和平移
        R = Ti0_list[i][:3, :3]  # 3×3旋转矩阵
        t = Ti0_list[i][:3, 3]   # 3维平移向量
        
        # 3. 变换点云到全局坐标系：p_global = R @ p_local + t
        # 注意：numpy广播，(3,3) @ (N_i,3).T = (3,N_i) → 转置回(N_i,3)
        global_pc = (R @ local_pc.T).T + t
        transformed_pieces.append(global_pc)
    
    # 4. 合并所有变换后的点云（拼接完整物体）
    full_pc = np.concatenate(transformed_pieces, axis=0)
    
    # 5. 转换为open3d点云格式（方便可视化/保存）
    full_pcd = o3d.geometry.PointCloud()
    full_pcd.points = o3d.utility.Vector3dVector(full_pc)
    
    return full_pcd, transformed_pieces

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
from open3d.web_visualizer import draw
# 模拟输入：3个碎片的局部点云 + 归一化后的变换矩阵Ti0
n = 3  # 碎片数量

# 1. 生成模拟碎片点云（0号是核心碎片，1/2号是周边碎片）
# 0号碎片：原点附近的立方体点云
piece0 = np.random.rand(1000, 3) - 0.5  # [1000,3], 范围[-0.5, 0.5]
# 1号碎片：局部坐标系在(1,0,0)附近，变换后对齐到0号右侧
piece1 = np.random.rand(800, 3) - 0.5   # [800,3]
# 2号碎片：局部坐标系在(0,1,0)附近，变换后对齐到0号上方
piece2 = np.random.rand(900, 3) - 0.5   # [900,3]
piece_point_clouds = [piece0, piece1, piece2]

# 2. 模拟归一化后的Ti0（相对0号的变换矩阵）
# 0号：自身变换为单位矩阵（归一化后固定）
T00 = np.eye(4)
# 1号：平移(0.8, 0, 0)，无旋转（相对0号右侧）
T10 = np.eye(4)
T10[:3, 3] = [0.8, 0, 0]
# 2号：平移(0, 0.8, 0)，旋转90°（相对0号上方）
T20 = np.eye(4)
T20[:3, 3] = [0, 0.8, 0]
T20[:3, :3] = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]])  # 绕x轴旋转90°
Ti0_list = [T00, T10, T20]

# 3. 执行拼接
full_pcd, transformed_pieces = assemble_pieces(piece_point_clouds, Ti0_list)

# 4. 可视化结果（对比原始碎片和拼接后）
# 原始碎片（不同颜色区分）
pcd0 = o3d.geometry.PointCloud()
pcd0.points = o3d.utility.Vector3dVector(piece0)
pcd0.paint_uniform_color([1, 0, 0])  # 0号：红色

pcd1 = o3d.geometry.PointCloud()
pcd1.points = o3d.utility.Vector3dVector(piece1)
pcd1.paint_uniform_color([0, 1, 0])  # 1号：绿色

pcd2 = o3d.geometry.PointCloud()
pcd2.points = o3d.utility.Vector3dVector(piece2)
pcd2.paint_uniform_color([0, 0, 1])  # 2号：蓝色

# 变换后碎片（同颜色，验证位置）
pcd0_transformed = o3d.geometry.PointCloud()
pcd0_transformed.points = o3d.utility.Vector3dVector(transformed_pieces[0])
pcd0_transformed.paint_uniform_color([1, 0, 0])

pcd1_transformed = o3d.geometry.PointCloud()
pcd1_transformed.points = o3d.utility.Vector3dVector(transformed_pieces[1])
pcd1_transformed.paint_uniform_color([0, 1, 0])

pcd2_transformed = o3d.geometry.PointCloud()
pcd2_transformed.points = o3d.utility.Vector3dVector(transformed_pieces[2])
pcd2_transformed.paint_uniform_color([0, 0, 1])

# 拼接后完整点云（灰色）
full_pcd.paint_uniform_color([0.5, 0.5, 0.5])

# 可视化：原始碎片（窗口1） + 变换后碎片（窗口2） + 拼接结果（窗口3）
draw([pcd0, pcd1, pcd2], title="原始碎片（局部坐标系）")
draw([pcd0_transformed, pcd1_transformed, pcd2_transformed], title="变换后碎片（全局坐标系）")
draw([full_pcd], title="拼接后的完整点云")

# 可选：保存拼接结果
# o3d.io.write_point_cloud("assembled_piece.ply", full_pcd)
# print("拼接完成，结果已保存为assembled_piece.ply")

[Open3D INFO] Window window_0 created.
[Open3D INFO] EGL headless mode enabled.
[Open3D INFO] ICE servers: {"stun:stun.l.google.com:19302", "turn:user:password@34.69.27.100:3478", "turn:user:password@34.69.27.100:3478?transport=tcp"}
[Open3D INFO] Set WEBRTC_STUN_SERVER environment variable add a customized WebRTC STUN server.
[Open3D INFO] WebRTC Jupyter handshake mode enabled.
FEngine (64 bits) created at 0x7ab86b042010 (threading is enabled)
EGL(1.5)
OpenGL(4.1)


WebVisualizer(window_uid='window_0')

[Open3D INFO] Window window_1 created.


WebVisualizer(window_uid='window_1')

[Open3D INFO] Window window_2 created.


WebVisualizer(window_uid='window_2')

拼接完成，结果已保存为assembled_piece.ply


In [18]:
import cv2
import numpy as np
import torch
import pickle as cPickle


def load_depth(img_path):
    """ Load depth image from img_path. """
    depth_path = img_path + '_depth.png'
    depth = cv2.imread(depth_path, -1)
    if len(depth.shape) == 3:
        # This is encoded depth image, let's convert
        # NOTE: RGB is actually BGR in opencv
        depth16 = depth[:, :, 1]*256 + depth[:, :, 2]
        depth16 = np.where(depth16==32001, 0, depth16)
        depth16 = depth16.astype(np.uint16)
    elif len(depth.shape) == 2 and depth.dtype == 'uint16':
        depth16 = depth
    else:
        assert False, '[ Error ]: Unsupported depth type.'
    return depth16

border_list = [-1, 40, 80, 120, 160, 200, 240, 280, 320, 360, 400, 440, 480, 520, 560, 600, 640, 680]
def get_bbox_from_mask(label, height = 480, width = 640):
    rows = np.any(label, axis=1)
    cols = np.any(label, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    rmax += 1
    cmax += 1
    r_b = rmax - rmin
    for tt in range(len(border_list)):
        if r_b > border_list[tt] and r_b < border_list[tt + 1]:
            r_b = border_list[tt + 1]
            break
    c_b = cmax - cmin
    for tt in range(len(border_list)):
        if c_b > border_list[tt] and c_b < border_list[tt + 1]:
            c_b = border_list[tt + 1]
            break

    center = [int((rmin + rmax) / 2), int((cmin + cmax) / 2)]

    rmin = center[0] - int(r_b / 2)
    rmax = center[0] + int(r_b / 2)
    cmin = center[1] - int(c_b / 2)
    cmax = center[1] + int(c_b / 2)

    if rmin < 0:
        delt = -rmin
        rmin = 0
        rmax += delt
    if cmin < 0:
        delt = -cmin
        cmin = 0
        cmax += delt
    if rmax > height:
        delt = rmax - height
        rmax = height
        rmin -= delt
    if cmax > width:
        delt = cmax - width
        cmax = width
        cmin -= delt
    return rmin, rmax, cmin, cmax

def get_instances_by_path(path, instance_id=-1, sample_num=2048, num_patches=15):
    # 从pkl文件加载数据，包括假定的detection结果和Pose等真值
    with open(path, 'rb') as f:
        pkl_data = cPickle.load(f)
        print(pkl_data.keys())
    image_path = pkl_data['image_path'] # os.path.join(self.data_dir, pred_data['image_path'][5:])
    # print(image_path)
    pred_mask = pkl_data['pred_masks']
    num_instance = len(pkl_data['pred_class_ids'])
    assert instance_id <= num_instance - 1
    assert type(instance_id) is int
    print(pkl_data['pred_class_ids'])
    # rgb
    rgb = cv2.imread(image_path + '_color.png')[:, :, :3]
    rgb = rgb[:, :, ::-1]  # 480*640*3
    img_h, img_w = rgb.shape[:2]
    # pts
    cam_fx, cam_fy, cam_cx, cam_cy = [665.80768, 665.80754, 637.642, 367.56]
    depth = load_depth(image_path)  # 480*640

    xmap = np.array([[i for i in range(img_w)] for _ in range(img_h)])
    ymap = np.array([[j for _ in range(img_w)] for j in range(img_h)])
    pts2 = depth.copy() / 1000.0
    pts0 = (xmap - cam_cx) * pts2 / cam_fx
    pts1 = (ymap - cam_cy) * pts2 / cam_fy
    pts = np.transpose(np.stack([pts0, pts1, pts2]), (1,2,0)).astype(np.float32)  # 480*640*3
    if instance_id == -1:
        all_rgb = []
        all_pts = []
        all_center = []
        all_cat_ids = []
        all_rgb_raw = []
        all_pts_raw = []
        all_mask = []
        all_choose = []

        flag_instance = torch.zeros(num_instance) == 1
        for j in range(num_instance):
            inst_mask = 255 * pred_mask[:, :, j].astype('uint8')
            mask = inst_mask > 0
            mask = np.logical_and(mask, depth>0)
            if np.sum(mask) > 16:
                rmin, rmax, cmin, cmax = get_bbox_from_mask(mask, img_h, img_w)
                cat_id = pkl_data['pred_class_ids'][j] - 1

                pts_raw = pts[rmin:rmax, cmin:cmax, :]
                
                rgb_raw = rgb[rmin:rmax, cmin:cmax, :]
                mask = mask[rmin:rmax, cmin:cmax]
                choose = mask.flatten().nonzero()[0]

                pts_raw = np.where((mask == 0)[:,:,None], np.nan, pts_raw)
                
                instance_pts = pts_raw.reshape((-1, 3))[choose, :].copy()
                
                rgb_raw = np.array(rgb_raw.copy()).astype(np.float32) / 255.0
                instance_rgb = rgb_raw.copy().reshape((-1, 3))[choose, :]
                
                center = np.mean(instance_pts, axis=0)
                # import pdb;pdb.set_trace()
                instance_pts = instance_pts - center[np.newaxis, :]
                pts_raw = pts_raw - center[np.newaxis, np.newaxis, :]
                if instance_pts.shape[0] <= sample_num:
                    choose_idx = np.random.choice(np.arange(instance_pts.shape[0]), sample_num)
                else:
                    choose_idx = np.random.choice(np.arange(instance_pts.shape[0]), sample_num, replace=False)
                instance_pts = instance_pts[choose_idx, :]
                instance_rgb = instance_rgb[choose_idx, :]

                rgb_raw = cv2.resize(rgb_raw, dsize=(num_patches*14, num_patches*14), interpolation=cv2.INTER_NEAREST)
                pts_raw = cv2.resize(pts_raw, dsize=(num_patches, num_patches), interpolation=cv2.INTER_NEAREST)
                mask = np.logical_not(np.isnan(pts_raw)).all(axis = -1)
                choose = mask.flatten().nonzero()[0]
                if len(choose)<=0:
                    continue
                elif len(choose) <= sample_num:
                    choose_idx = np.random.choice(np.arange(len(choose)), sample_num)
                else:
                    choose_idx = np.random.choice(np.arange(len(choose)), sample_num, replace=False)
                choose = choose[choose_idx]

                all_pts_raw.append(torch.FloatTensor(pts_raw))
                all_rgb_raw.append(torch.FloatTensor(rgb_raw))
                all_choose.append(torch.IntTensor(choose).long())
                all_mask.append(torch.IntTensor(mask).long())
                all_pts.append(torch.FloatTensor(instance_pts))
                all_rgb.append(torch.FloatTensor(instance_rgb))
                all_center.append(torch.FloatTensor(center))
                all_cat_ids.append(torch.IntTensor([cat_id]).long())
                flag_instance[j] = 1

        ret_dict = {}

        ret_dict['gt_class_ids'] = torch.tensor(pkl_data['gt_class_ids'])
        ret_dict['gt_bboxes'] = torch.tensor(pkl_data['gt_bboxes'])
        ret_dict['gt_RTs'] = torch.tensor(pkl_data['gt_RTs'])
        ret_dict['gt_scales'] = torch.tensor(pkl_data['gt_scales'])
        ret_dict['gt_handle_visibility'] = torch.tensor(pkl_data['gt_handle_visibility'])
        ## 新增对装配特征匹配对的ground truth
        ret_dict['gt_match'] = torch.tensor(pkl_data['gt_match_pairs'])

        if len(all_pts) == 0:
            ret_dict['pred_class_ids'] = torch.tensor(pkl_data['pred_class_ids'])
            ret_dict['pred_bboxes'] = torch.tensor(pkl_data['pred_bboxes'])
            ret_dict['pred_scores'] = torch.tensor(pkl_data['pred_scores'])

        else:
            ret_dict['pts'] = torch.stack(all_pts) # N*3
            ret_dict['rand_rotation'] = torch.eye(3)[None,:,:].repeat(ret_dict['pts'].shape[0],1,1)
            ret_dict['rgb'] = torch.stack(all_rgb)
            ret_dict['pts_raw'] = torch.stack(all_pts_raw) # N*3
            ret_dict['rgb_raw'] = torch.stack(all_rgb_raw)
            ret_dict['choose'] = torch.stack(all_choose)
            ret_dict['mask'] = torch.stack(all_mask)
            ret_dict['center'] = torch.stack(all_center)
            ret_dict['category_label'] = torch.stack(all_cat_ids).squeeze(1)
            ret_dict['pred_class_ids'] = torch.tensor(pkl_data['pred_class_ids'])[flag_instance==1]
            ret_dict['pred_bboxes'] = torch.tensor(pkl_data['pred_bboxes'])[flag_instance==1]
            ret_dict['pred_scores'] = torch.tensor(pkl_data['pred_scores'])[flag_instance==1]

        return ret_dict

In [49]:
from model.jigsaw.joint_seg_align_model import JointSegmentationAlignmentModel as JigsawModel
from model import build_model
from utils.config import cfg_from_file
from pytorch3d.transforms import matrix_to_quaternion
cfg_from_file('experiments/jigsaw_250e_cosine.yaml')
from utils.config import cfg
res = get_instances_by_path('/data/yan/pose_dataset/multi_asm/NOCS/detection/results_000001_000006.pkl')
print(res.keys())
cls_ids = res.get('category_label') + 1
pts = res.get('pts', None)
print(f"{pts.max()}, {pts.min()}")
print(cls_ids)
src_pc_clsMask = cls_ids == 3
pc1 = pts[src_pc_clsMask]
print(pc1.shape)
target_pc_clsMask = cls_ids == 4
pc2 = pts[target_pc_clsMask]
print(pc2.shape)

n_pcs = pts.shape[1] * torch.ones(pts.shape[0], dtype=torch.long).view(1, -1)
print(n_pcs.shape)
n_valid = torch.tensor(pts.shape[0], dtype=torch.long).view(1, -1)
print(n_valid)
model = build_model(cfg)
# model.eval()
labels = model.compute_label(pts.view(1, -1, 3), n_pcs, n_valid, 0.0025)

pose = torch.tensor(res['gt_RTs'])
quats_gt = matrix_to_quaternion(pose[:, :3, :3])
trans_gt = pose[:, :3, 3]
data_dict = {
    "part_pcs": pts.view(1, -1, 3),
    "gt_pcs": pts.view(1, -1, 3),
    "part_valids": torch.ones((1, pts.shape[0])),
    "part_quat": quats_gt,
    "part_trans": trans_gt,
    "n_pcs": n_pcs,
    "data_id": 0,
    "critical_label_thresholds": 0.0025,
}

mf = model.forward(data_dict)
print(mf)

dict_keys(['image_path', 'pred_class_ids', 'pred_masks', 'pred_bboxes', 'pred_scores', 'pred_scales', 'gt_class_ids', 'gt_bboxes', 'gt_scales', 'gt_RTs', 'gt_match_pairs', 'gt_handle_visibility'])
[4 4 4 4 3 3 3 3]
dict_keys(['gt_class_ids', 'gt_bboxes', 'gt_RTs', 'gt_scales', 'gt_handle_visibility', 'gt_match', 'pts', 'rand_rotation', 'rgb', 'pts_raw', 'rgb_raw', 'choose', 'mask', 'center', 'category_label', 'pred_class_ids', 'pred_bboxes', 'pred_scores'])
0.6624851226806641, -0.30849599838256836
tensor([4, 4, 4, 4, 3, 3, 3, 3])
torch.Size([4, 2048, 3])
torch.Size([4, 2048, 3])
torch.Size([1, 8])
tensor([[8]])
initial: self.w_cls_loss 1.0, self.w_mat_loss 0.0, self.w_rig_loss 0.0
self.pc_cls_method: binary


/tmp/ipykernel_1904788/301294142.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pose = torch.tensor(res['gt_RTs'])


{'cls_logits': tensor([[[ 0.6480],
         [ 0.2889],
         [ 0.1939],
         ...,
         [-0.1916],
         [-0.4896],
         [ 0.2711]]], grad_fn=<TransposeBackward0>), 'cls_pred': tensor([[1, 1, 1,  ..., 0, 0, 1]]), 'batch_size': 1}
